<a href="https://colab.research.google.com/github/nithinrb/NVIDIA/blob/main/DAY_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# TOPIC 1 - PROMPT ENGINEERING
# Same task asked three ways, so students see better prompts give better answers.

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map="auto")

def ask(text, n=120):
    msg = [{"role": "user", "content": text}]
    p = tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    ids = tok(p, return_tensors="pt").to(model.device)
    out = model.generate(**ids, max_new_tokens=n, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# Zero-shot: just ask
print("Zero-shot:", ask("Is this review positive or negative? 'Broke in two days.'", 10))

# Few-shot: show the pattern first
print("Few-shot :", ask("good->Positive  bad->Negative  'fast and reliable'->", 5))

# Chain-of-thought: ask it to think step by step (gives smarter answers)
print("Step-by-step:", ask("Solve step by step: a train goes 60km in 1.5h. Average speed?"))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Zero-shot: The review "Broke in two days." is
Few-shot : Positive
Step-by-step: To find the average speed of the train, we use the formula for average speed:

\[ \text{Average Speed} = \frac{\text{Total Distance}}{\text{Total Time}} \]

Given:
- Total Distance = 60 km
- Total Time = 1.5 hours

Now plug these values into the formula:

\[ \text{Average Speed} = \frac{60 \text{ km}}{1.5 \text{ h}} \]

Let's do this division step by step.

First, divide the numerator (top number) by the denominator


In [2]:
# TOPIC 2 - HUGGING FACE ON NVIDIA H200
# Load a model on the GPU and see how much memory it uses.

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

# Check the GPU (on the lab box this shows the H200 with ~141 GB)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map="auto")

# How much GPU memory the model takes
if torch.cuda.is_available():
    print("Model uses about", round(torch.cuda.memory_allocated()/1e9, 2), "GB of GPU memory")

# Generate one reply
msg = [{"role": "user", "content": "Explain what a GPU does in one sentence."}]
p = tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
ids = tok(p, return_tensors="pt").to(model.device)
out = model.generate(**ids, max_new_tokens=60, pad_token_id=tok.eos_token_id)
print("Reply:", tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip())

# The H200's big 141 GB memory is what lets you run bigger models and serve
# many users at once without running out of space.


GPU: Tesla T4


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model uses about 3.1 GB of GPU memory
Reply: A Graphics Processing Unit (GPU) is designed to perform complex mathematical operations and parallel processing tasks quickly, making it ideal for rendering graphics and handling large data sets in applications like video games, scientific simulations, and deep learning.


In [8]:
# TOPIC 3 - LoRA / PEFT FINE-TUNING
# Teach the model a new trick by training only a tiny number of parameters.

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.pad_token or tok.eos_token

# Load the model in 4-bit so it fits easily (the "Q" in QLoRA)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="auto")
model = prepare_model_for_kbit_training(model)

# Attach small LoRA adapters (the sticky notes); only these get trained
model = get_peft_model(model, LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
                       target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM"))
model.print_trainable_parameters()   # shows under 1% of parameters are trained

# A small set of example instructions to learn from
data = load_dataset("yahma/alpaca-cleaned", split="train[:200]").map(
    lambda e: {"text": f"### Instruction:\n{e['instruction']}\n\n### Response:\n{e['output']}"})

# Train, then save the tiny adapter (only a few MB)
SFTTrainer(model=model, train_dataset=data, processing_class=tok,
    args=SFTConfig(output_dir="./out", num_train_epochs=1, per_device_train_batch_size=4,
                   learning_rate=2e-4, bf16=True,
                   dataset_text_field="text", report_to="none")).train()
model.save_pretrained("./day7-adapter")
print("Done - adapter saved.")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


Adding EOS to train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,2.052620
20,1.696642
30,1.626189
40,1.744117
50,1.243596


Done - adapter saved.


In [10]:
!pip install -U trl transformers peft accelerate bitsandbytes datasets torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 97.9 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [12]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='torchao')

# TOPIC 3 (part 2) - USE THE TRAINED ADAPTER
# Run this after 3_lora_finetuning has saved ./day7-adapter

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)

# Load the big base model, then stick the tiny trained adapter on top
base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map="auto")
model = PeftModel.from_pretrained(base, "./day7-adapter")

# Ask it something
prompt = "### Instruction:\nExplain what a GPU does in one sentence.\n\n### Response:\n"
ids = tok(prompt, return_tensors="pt").to(model.device)
out = model.generate(**ids, max_new_tokens=80, pad_token_id=tok.eos_token_id)
print("Reply:", tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip())

# Optional: merge the adapter into the base model for fast deployment
merged = model.merge_and_unload()
print("Adapter merged into the model - ready to deploy.")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Reply: A GPU, or graphics processing unit, is designed to perform complex mathematical and computational tasks at high speeds for rendering images and handling video games.
Adapter merged into the model - ready to deploy.
